# 00 - Setup and verification

Run this first, once per Colab session. It:
1. connects your Google Drive,
2. installs the libraries,
3. checks that the masks from the earlier step are where we expect,
4. measures the two models (parameters, FLOPs, inference time) for the Part B table.

**What you should see:** your GPU name, the masks-index preview (8000 train / 1000 val / 1000 test), and a small table about each model.

### 1. Connect Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Install libraries

In [2]:
!pip install -q segmentation-models-pytorch transformers optuna thop joblib
print('installs done')

installs done


### 3. Import our shared code
All the project's code lives in `code/` inside this package (in my Drive). These notebook cells call into it, so the same pipeline drives every experiment.

In [3]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
import torch
print('torch', torch.__version__)
C.save_config()   # the reproducibility record, saved to results/config.json
print('config saved to', C.RESULTS_DIR)

GPU: Tesla T4
torch 2.14.1+cu130
config saved to /content/drive/MyDrive/coco_seg/project/results


### 4. Verify the masks and splits
This reads `masks_index.csv` (written by the masks walkthrough) and matches every photo to its mask. **Expected:** 8000 train, 1000 val, 1000 test, 0 missing files.

In [4]:
frames = data.load_frames()
for split, f in frames.items():
    print(f'{split:5s}: {len(f)} images')
assert len(frames['train']) == 8000, 'expected 8000 train'
assert len(frames['val']) == 1000 and len(frames['test']) == 1000, 'expected 1000/1000'
print('Masks and splits verified.')

masks index: 10000 rows | splits: {'train': 8000, 'val': 1000, 'test': 1000} | missing files: 0
                                                                            image                                                          mask  split
0  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000307967.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000307967.png  train
1  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000154345.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000154345.png  train
train: 8000 images
val  : 1000 images
test : 1000 images
Masks and splits verified.


### 5. Model cards (Part B-1 numbers)
Downloads the pretrained backbone models (once, a few hundred MB) and evaluates each one. This completes the *parameters/FLOPs/inference time* table requested in the prompt.

In [5]:
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
cards = {}
for name in C.MODELS:
    print(f'--- {name} ---')
    cards[name] = models.model_card(name, device=device)
    for k, v in cards[name].items():
        print(f'  {k}: {v}')
report_utils.save_result('models', cards)
print('saved results/models.json')

--- deeplabv3plus ---


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

  model: deeplabv3plus
  input_size: 512
  params_total: 26680669
  params_trainable: 26680669
  gflops: 36.988783616
  inference_ms_per_image: 23.897559233334203
--- segformer_b3 ---


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

... (4 outputs left out)

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b3
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.batch_norm.num_batches_tracked              

model.safetensors: reconstructing file:   0%|          |  0.00B /  178MB            

model.safetensors: downloading bytes:           |  0.00B            

  model: segformer_b3
  input_size: 512
  params_total: 47232461
  params_trainable: 47232461
  gflops: 71.499055104
  inference_ms_per_image: 62.47206359999875
saved results/models.json


In [6]:

import os, glob
print('Drive mounted:', os.path.exists('/content/drive/MyDrive'))
p = '/content/drive/MyDrive/coco_seg'
print('coco_seg contains:', os.listdir(p) if os.path.exists(p) else 'MISSING')
print(glob.glob('/content/drive/MyDrive/**/config.py', recursive=True))

Drive mounted: True
coco_seg contains: ['fiftyone', 'project', 'masks', 'masks_index.csv', 'class_pixel_distribution.csv', 'class_distribution.png', 'dataset_summary.csv']
['/content/drive/MyDrive/coco_seg/project/code/config.py']
